### Line 8 with random diagonal and anti-diagonal 

In [ ]:
import os, shutil, random
import pydicom, hashlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
from pathlib import Path
from PIL import Image
from concurrent.futures import ThreadPoolExecutor, as_completed

In [ ]:
WORK_DIR = Path("/home/jupyter-yin10/Image_Analysis")
IMG_DIR  = Path("/data0/NIH-CXR14/images")
RAW_CSV  = Path("/data0/NIH-CXR14/Data_Entry_2017_v2020.csv")

# ================= output files =================
SAMPLE_CSV        = WORK_DIR / "meta_NIH-CXR14.csv"
SAMPLE_WITH_PATHS = WORK_DIR / "NIH-CXR14_with_paths.csv"
out_root          = WORK_DIR / "random_l8"   

In [ ]:
SEED = 42
WORK_DIR.mkdir(parents=True, exist_ok=True)

SQUARE_SIZE = 8

In [ ]:
df_meta = pd.read_csv(RAW_CSV)
print("Meta data shape: {}".format(df_meta.shape))
print("\n")
print("Colums: {}".format(list(df_meta.columns)))
print("\n")
df_meta.to_csv(SAMPLE_CSV, index=False)
df_meta.head()

# add image index column
df = pd.read_csv("/home/jupyter-yin10/Image_Analysis/meta_NIH-CXR14.csv")
df["Image Path"] = "/data0/NIH-CXR14/images/" + df["Image Index"]
df.to_csv("/home/jupyter-yin10/Image_Analysis/meta_NIH-CXR14_with_paths.csv")
print("Dataset shape: {}".format(df.shape))
print("\n")
print(df.head())

In [ ]:
# ====================== helper functions ======================
def _seed_from_name(name: str, base_seed: int) -> int:
    # Stable per-image seed from base SEED and a stable string
    h = hashlib.sha256()
    h.update(str(base_seed).encode("utf-8"))
    h.update(b"::")
    h.update(name.encode("utf-8"))
    return int.from_bytes(h.digest()[:8], "little", signed=False)

def choose_random_top_left(img_path: Path, tile_size: int, base_seed: int, key_str: str) -> tuple[int, int]:
    """Pick a (x, y) top-left so a tile of size tile_size × tile_size fits."""
    with Image.open(img_path) as im:
        w, h = im.size  # (width, height)
    s = tile_size
    xmin, xmax = 0, max(0, w - s)
    ymin, ymax = 0, max(0, h - s)
    if xmax < xmin or ymax < ymin:
        return (0, 0)
    seed = _seed_from_name(key_str, base_seed)
    rng = random.Random(seed)
    x = rng.randint(xmin, xmax)
    y = rng.randint(ymin, ymax)
    return (x, y)

def compute_intensity_range(img_path: Path, span: int = 20) -> tuple[int, int, int]:
    """
    Find grayscale histogram mode x in [0..255], define [x-span, x+span] clamped to [0,255].
    Returns (lo, hi, x). No RNG here; just image content.
    """
    with Image.open(img_path) as img:
        img = img.convert("L")
        hist = img.histogram()[:256]
    x  = int(np.argmax(hist))
    lo = max(0, x - span)
    hi = min(255, x + span)
    return lo, hi, x

def choose_diagonal_type(base_seed: int, key_str: str) -> str:
    """
    Deterministic per-image choice between 'diagonal' and 'anti_diagonal'
    based on your seeded hash. 50/50-ish split across many images.
    """
    s = _seed_from_name("diagtype::" + key_str, base_seed)
    return "anti_diagonal" if (s % 2) else "diagonal"

def add_diagonal_8x8(img_path: Path,
                     out_path: Path,
                     top_left: tuple[int,int],
                     base_seed: int,
                     key_str: str,
                     lo: int,
                     hi: int,
                     diagonal_type: str):

    with Image.open(img_path) as img:
        img = img.convert("L")
        arr = np.array(img)

    H, W = arr.shape
    x, y = top_left
    s = 8
    if x < 0 or y < 0 or x + s > W or y + s > H:
        out_path.parent.mkdir(parents=True, exist_ok=True)
        Image.fromarray(arr).save(out_path)
        return

    # deterministic per-image RNG for the two diagonal pixels
    pix_seed = _seed_from_name("pixels::" + key_str, base_seed)
    rng = np.random.default_rng(pix_seed)

    if diagonal_type == "anti_diagonal":
        coords = [(y + 0, x + 1), (y + 1, x + 0)]
    else:  # "diagonal"
        coords = [(y + 0, x + 0), (y + 1, x + 1)]

    for (yy, xx) in coords:
        v = int(rng.integers(lo, hi + 1))
        arr[yy, xx] = v

    out_path.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray(arr).save(out_path)

def copy_or_link(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    try:
        os.link(src, dst)
    except OSError:
        try:
            os.symlink(src, dst)
        except OSError:
            shutil.copyfile(src, dst)

            
def _process_row(r, split: str):
    src = Path(r["Image Path"])
    dst = out_root / split / r["label"] / Path(r["Image Index"]).name

    if r["label"] == "dot":
        base_seed = SEED if split == "train" else SEED + 1

        # 1) pick top-left for the 8×8 tile
        top_left = choose_random_top_left(
            img_path=src,
            tile_size=SQUARE_SIZE,
            base_seed=base_seed,
            key_str=str(dst.name)
        )

        # 2) histogram-informed intensity band (mode ± 20)
        lo, hi, _mode = compute_intensity_range(src)

        # 3) deterministic per-image choice of diagonal orientation
        diag_type = choose_diagonal_type(base_seed, str(dst.name))

        # 4) write exactly the diagonal pixels of the 8×8
        add_diagonal_8x8(
            img_path=src,
            out_path=dst,
            top_left=top_left,
            base_seed=base_seed,
            key_str=str(dst.name),
            lo=lo,
            hi=hi,
            diagonal_type=diag_type
        )
    else:
        copy_or_link(src, dst)

def materialize_split_parallel(dframe: pd.DataFrame, split: str, workers: int = 8, log_every: int = 2000):
    total = len(dframe)
    with ThreadPoolExecutor(max_workers=workers) as ex:
        futs = [ex.submit(_process_row, r, split) for _, r in dframe.iterrows()]
        for i, f in enumerate(as_completed(futs), 1):
            _ = f.result()  # raise early if any task failed
            if i % log_every == 0 or i == total:
                print(f"[{split}] {i}/{total} ({i/total:.1%})")

def label_balanced(dframe: pd.DataFrame, seed: int):
    n = len(dframe)
    idx = list(range(n))
    random.Random(seed).shuffle(idx)
    half = n // 2
    with_dot = set(idx[:half])  # half with dot, half with nodot
    dframe["label"] = ["dot" if i in with_dot else "nodot" for i in range(n)]
    return dframe


In [ ]:
df = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
n_total = len(df)
n_train = int(n_total * 0.8)
train_df = df.iloc[:n_train].copy()
test_df  = df.iloc[n_train:].copy()

train_df = label_balanced(train_df, SEED)
test_df  = label_balanced(test_df,  SEED)

print("Train label counts: \n{}.".format(train_df["label"].value_counts()))
print("\n")
print("Test label counts: \n{}.".format(test_df["label"].value_counts()))


In [ ]:
shutil.rmtree(out_root, ignore_errors=True)
out_root.mkdir(parents=True, exist_ok=True)

materialize_split_parallel(train_df, "train")
materialize_split_parallel(test_df,  "test")

print("Done with written images to {}".format(out_root))

In [ ]:
def count_pngs(folder: Path):
    return sum(1 for _ in folder.glob("*.png"))

train_dot   = out_root / "train" / "dot"
train_nodot = out_root / "train" / "nodot"
test_dot    = out_root / "test"  / "dot"
test_nodot  = out_root / "test"  / "nodot"

print("Counts:")
print("Train/dot: {}". format(count_pngs(train_dot)))
print("Train/nodot: {}". format(count_pngs(train_nodot)))
print("Test/dot: {}". format(count_pngs(test_dot)))
print("Test/nodot: {}". format(count_pngs(test_nodot)))

for i in [train_dot, test_dot]:
    files = list(i.glob("*.png"))
    if files:
        img = Image.open(files[0])
        print("{} first image size (width, height) is: {}".format(i, img.size))
    else:
        print("No images found in {}".format(i))

In [ ]:
dot_images = list((out_root / "train" / "dot").glob("*.png"))
if len(dot_images) == 0:
    raise RuntimeError("No dot images found in train/dot to verify.")

img_path = dot_images[1] if len(dot_images) > 1 else dot_images[0]

# recompute same placement + orientation deterministically
base_seed = SEED
x, y = choose_random_top_left(
    img_path=img_path,
    tile_size=SQUARE_SIZE,
    base_seed=base_seed,
    key_str=str(img_path.name)
)
lo, hi, mode_x = compute_intensity_range(img_path)
diag_type = choose_diagonal_type(base_seed, str(img_path.name))

img = Image.open(img_path).convert("L")
arr = np.array(img)

if diag_type == "anti_diagonal":
    coords = [(y+0, x+1), (y+1, x+0)]
else:
    coords = [(y+0, x+0), (y+1, x+1)]

obs_vals = [int(arr[yy, xx]) for (yy, xx) in coords]

print(f"8×8 tile top-left at (x,y)=({x}, {y})  |  diag_type: {diag_type}")
print(f"Histogram mode: {mode_x}, range used: [{lo}, {hi}]")
print("Observed diagonal intensities (8 pixels):", obs_vals)

# draw a red rectangle around the 2×2 tile and a red line on the chosen diagonal
fig, ax = plt.subplots(1, 1, figsize=(7, 7))
ax.imshow(arr, cmap="gray")
ax.add_patch(Polygon(
    [(x-0.5, y-0.5), (x+SQUARE_SIZE-0.5, y-0.5),
     (x+SQUARE_SIZE-0.5, y+SQUARE_SIZE-0.5), (x-0.5, y+SQUARE_SIZE-0.5)],
    fill=False, edgecolor="red", linewidth=2
))
# diagonal overlay for visualization
if diag_type == "anti_diagonal":
    ax.plot([x+SQUARE_SIZE-0.5, x-0.5], [y-0.5, y+SQUARE_SIZE-0.5], linewidth=2, color="red")
else:
    ax.plot([x-0.5, x+SQUARE_SIZE-0.5], [y-0.5, y+SQUARE_SIZE-0.5], linewidth=2, color="red")
ax.set_title(f"8×8 square {diag_type} line")
ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
root = out_root
LABEL_MAP = {"dot": 1, "nodot" :0}

def make_manifest(split:str, root: Path = root, label_map: dict = LABEL_MAP, pattern: str = "*.png") -> pd.DataFrame:
    rows =[]
    for label_name, label_id in label_map.items():
        folder = root / split / label_name
        if not folder.exists():
            print("Folder not found.")
            continue
        for j in sorted(folder.glob(pattern)):
            rows.append({"path":str(j),"label":int(label_id)})
    return pd.DataFrame(rows, columns=["path","label"])

train_df = make_manifest("train")
test_df  = make_manifest("test")

train_manifest_path = root / "train_manifest.csv"
test_manifest_path  = root / "test_manifest.csv"
train_df.to_csv(train_manifest_path, index=False)
test_df.to_csv(test_manifest_path, index=False)

print("Saved train manifest to {}".format(train_manifest_path))
print("Saved test manifest to {}".format(test_manifest_path))
print("Train rows: ", len(train_df), " | dot: ", (train_df['label']==1).sum(), " | nodot: ", (train_df['label']==0).sum())
print("Test rows: ", len(test_df), " | dot: ", (test_df['label']==1).sum(), " | nodot: ", (test_df['label']==0).sum())
print("\nPreview:\n", train_df.head())

In [ ]:
# === Diagonal vs Anti-diagonal counts for dot *and* nodot ===
from pathlib import Path
import pandas as pd, hashlib

# Reuse globals if present; otherwise set sensible defaults
try:
    ROOT = out_root
except NameError:
    ROOT = Path("/home/jupyter-yin10/Image_Analysis/random_l8")

try:
    TRAIN_SEED = SEED
except NameError:
    TRAIN_SEED = 42
TEST_SEED = TRAIN_SEED + 1

try:
    _seed_from_name
except NameError:
    def _seed_from_name(name: str, base_seed: int) -> int:
        h = hashlib.sha256()
        h.update(str(base_seed).encode("utf-8"))
        h.update(b"::")
        h.update(name.encode("utf-8"))
        return int.from_bytes(h.digest()[:8], "little", signed=False)

try:
    choose_diagonal_type
except NameError:
    def choose_diagonal_type(base_seed: int, key_str: str) -> str:
        s = _seed_from_name("diagtype::" + key_str, base_seed)
        return "anti_diagonal" if (s % 2) else "diagonal"

def count_by_split_label(root: Path = ROOT, train_seed: int = TRAIN_SEED, test_seed: int = TEST_SEED):
    rows = []
    for split, seed in [("train", train_seed), ("test", test_seed)]:
        for label in ["dot", "nodot"]:
            folder = root / split / label
            if not folder.exists():
                continue
            files = list(folder.glob("*.png"))
            total = len(files)
            if label == "dot":
                diag = sum(choose_diagonal_type(seed, p.name) == "diagonal" for p in files)
                anti = total - diag
            else:
                # nodot images have no diagonal/anti-diagonal noise
                diag = 0
                anti = 0
            rows.append({
                "split": split,
                "label": label,
                "count": total,
                "diagonal": diag,
                "anti_diagonal": anti
            })
    if not rows:
        print("No images found under", root)
        return None, None

    detail = pd.DataFrame(rows).sort_values(["split", "label"]).reset_index(drop=True)

    # Totals per split (dot + nodot)
    split_totals = (
        detail.groupby("split")[["count", "diagonal", "anti_diagonal"]]
        .sum().reset_index()
    )
    split_totals["label"] = "ALL"

    # Grand total across splits
    grand = detail[["count", "diagonal", "anti_diagonal"]].sum().to_dict()
    grand_row = {"split": "ALL", "label": "ALL", **grand}

    with_totals = pd.concat([detail, split_totals, pd.DataFrame([grand_row])],
                            ignore_index=True)

    return detail, with_totals

detail_df, totals_df = count_by_split_label()

print("\nDetail by split & label:")
print(detail_df)

print("\nWith split totals and grand total:")
print(totals_df)

# Quick sanity checks
if detail_df is not None:
    for split in ["train", "test"]:
        try:
            dot_row = detail_df[(detail_df.split == split) & (detail_df.label == "dot")].iloc[0]
            ok = dot_row["count"] == dot_row["diagonal"] + dot_row["anti_diagonal"]
            print(f"{split}: dot == diagonal+anti ? {ok}")
        except IndexError:
            pass
